# Waldo AI Exploration
Notebook de exploração do pipeline: geração de cena, bbox, dataset sintético e inferência YOLO.

In [ ]:
from pathlib import Path
import sys
import random

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

%matplotlib inline

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'backend').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from backend.scene_generation.generate_scene import generate_scene
from backend.scene_generation.characters import waldo_sprite, random_character_sprite
from backend.scene_generation.collision import area, overlap_area, can_place_bbox
from backend.utils.bbox_utils import bbox_to_yolo, yolo_to_bbox, point_in_bbox, iou
from backend.utils.config import DIFFICULTY_TO_COUNT, get_paths
from backend.vision.dataset.dataset_builder import generate_dataset
from backend.vision.inference.detect_waldo import detect_waldo

paths = get_paths()
paths

## Generate and Inspect a Scene

In [ ]:
scene, waldo_bbox = generate_scene(difficulty='medium', seed=42)
scene_array = np.array(scene)
x1, y1, x2, y2 = waldo_bbox

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(scene_array)
ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='lime', facecolor='none'))
ax.set_title('Generated scene with ground-truth Waldo box')
ax.axis('off')
plt.show()

print('Ground truth bbox:', waldo_bbox)
print('YOLO normalized bbox:', bbox_to_yolo(waldo_bbox, scene.width, scene.height))

## Difficulty Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, difficulty in zip(axes, ['easy', 'medium', 'hard']):
    image, bbox = generate_scene(difficulty=difficulty, seed=7)
    x1, y1, x2, y2 = bbox
    ax.imshow(np.array(image))
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='cyan', facecolor='none'))
    ax.set_title(f'{difficulty} | crowd={DIFFICULTY_TO_COUNT[difficulty]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Sprite Inspection

In [ ]:
rng = random.Random(0)
sizes = [(32, 56), (48, 84), (64, 112)]
waldos = [waldo_sprite(rng, size) for size in sizes]
chars = [random_character_sprite(rng, size) for size in sizes]

fig, axes = plt.subplots(2, len(sizes), figsize=(12, 6))
for col, (size, sprite) in enumerate(zip(sizes, waldos)):
    axes[0, col].imshow(sprite)
    axes[0, col].set_title(f'Waldo {size}')
    axes[0, col].axis('off')
for col, (size, sprite) in enumerate(zip(sizes, chars)):
    axes[1, col].imshow(sprite)
    axes[1, col].set_title(f'Character {size}')
    axes[1, col].axis('off')
plt.tight_layout()
plt.show()

## Bounding Box Utilities

In [ ]:
sample_bbox = waldo_bbox
yolo_bbox = bbox_to_yolo(sample_bbox, scene.width, scene.height)
restored_bbox = yolo_to_bbox(*yolo_bbox, scene.width, scene.height)

print('Pixel bbox       :', sample_bbox)
print('YOLO bbox        :', yolo_bbox)
print('Restored pixel   :', restored_bbox)
print('Inside center?   :', point_in_bbox((sample_bbox[0] + sample_bbox[2]) / 2, (sample_bbox[1] + sample_bbox[3]) / 2, sample_bbox))
print('Self IoU         :', iou(sample_bbox, restored_bbox))

## Collision Utilities

In [ ]:
box_a = (10, 10, 60, 80)
box_b = (40, 30, 95, 90)
box_c = (120, 120, 180, 200)

print('Area A           :', area(box_a))
print('Area B           :', area(box_b))
print('Overlap A/B      :', overlap_area(box_a, box_b))
print('Can place C      :', can_place_bbox(box_c, [box_a, box_b]))
print('Can place B again:', can_place_bbox(box_b, [box_a]))

## Small Dataset Generation Smoke Test

In [ ]:
summary = generate_dataset(n_images=6, seed=123)
summary

## YOLO Inference

In [ ]:
detections = detect_waldo(scene)
if not detections:
    print('No trained model found yet in frontend/models. Train the model first to enable inference.')
else:
    best = detections[0]
    pred_bbox = best['bbox']
    conf = best['confidence']
    print('Best detection:', best)
    print('IoU vs GT:', iou(waldo_bbox, pred_bbox))

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(np.array(scene))
    ax.add_patch(patches.Rectangle((waldo_bbox[0], waldo_bbox[1]), waldo_bbox[2] - waldo_bbox[0], waldo_bbox[3] - waldo_bbox[1], linewidth=2, edgecolor='lime', facecolor='none'))
    ax.add_patch(patches.Rectangle((pred_bbox[0], pred_bbox[1]), pred_bbox[2] - pred_bbox[0], pred_bbox[3] - pred_bbox[1], linewidth=2, edgecolor='red', facecolor='none'))
    ax.set_title(f'GT (green) vs YOLO (red) | conf={conf:.3f}')
    ax.axis('off')
    plt.show()